# CT 助手 LoRA / SFT 训练与独立测试

在 **代码执行程序 → 更改运行时类型 → T4 GPU** 中选择 GPU，保存一份 Drive 副本后运行全部。

基于 Qwen2.5-1.5B-Instruct，训练工具调用与证据选择。**不训练 CT 检测器，不证明真实工业缺陷/根因识别准确率。**

24 个训练、6 个验证、6 个测试合成样本，按完整样本分割。比较无格式约束的单步输出，再验证受约束 Agent 的完整流程。基础模型和 Adapter 使用相同测试集、贪心解码。

In [1]:
import os, subprocess, sys
from pathlib import Path
repo = Path('/content/xsim-chip-lora')
if not repo.exists():
    subprocess.run(['git', 'clone', 'https://github.com/yuweimin2077-hub/xsim-chip.git', str(repo)], check=True)
os.chdir(repo)
print('Code revision:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[training]'], check=True)
import torch
assert torch.cuda.is_available(), 'Select a GPU runtime before training'
print('GPU:', torch.cuda.get_device_name(0), 'Torch:', torch.__version__)

Code revision: 11f5f816c37aff0217bc5a9b147597143fb6f1cb
GPU: Tesla T4 Torch: 2.11.0+cu130


## 固定配置训练与测试

训练只监督最后一条助手动作；其余输入全部屏蔽 loss。超长证据拒绝截断。两轮训练结束保存 Adapter，然后重新加载它完成对照评测。此步骤可能需要十几分钟或更久。

In [2]:
subprocess.run([sys.executable, 'tools/train_assistant_lora.py', '--output-dir', 'outputs/lora_pilot'], check=True)

CompletedProcess(args=['/usr/bin/python3', 'tools/train_assistant_lora.py', '--output-dir', 'outputs/lora_pilot'], returncode=0)

## 查看结果并导出可复核文件

In [3]:
import json, hashlib, zipfile, base64
from IPython.display import display, HTML
out = repo / 'outputs/lora_pilot'
training = json.loads((out / 'training.json').read_text())
evaluation = json.loads((out / 'evaluation.json').read_text())
print(json.dumps({'gpu':training['gpu'], 'trainable_parameters':training['trainable_parameters'],
                  'validation_loss_before':training['baseline_validation_loss'],
                  'validation_loss_after':training['adapter_validation_loss'],
                  'training_peak_gib':training['training_peak_allocated_gib'],
                  'evaluation':evaluation}, ensure_ascii=False, indent=2))
files = [p for p in out.rglob('*') if p.is_file() and 'trainer' not in p.relative_to(out).parts]
integrity = {str(p.relative_to(out)): hashlib.sha256(p.read_bytes()).hexdigest() for p in files}
(out / 'integrity.json').write_text(json.dumps(integrity, indent=2) + '\n')
archive = repo / 'outputs/xsim_lora_pilot.zip'
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in files + [out / 'integrity.json']:
        z.write(p, str(p.relative_to(out)))
print('Archive bytes:', archive.stat().st_size, 'SHA256:', hashlib.sha256(archive.read_bytes()).hexdigest())
display(HTML('<a download="xsim_lora_pilot.zip" href="data:application/zip;base64,' + base64.b64encode(archive.read_bytes()).decode() + '">Download adapter, dataset, logs and evaluation ZIP</a>'))

Output hidden; open in https://colab.research.google.com to view.

## 实测结论：LoRA 训练完成（2026-10-04）

Qwen2.5-1.5B-Instruct · Tesla T4 · LoRA r=8 · 1,089,536 可训练参数 · 2 epochs

| 指标 | 基础模型 | LoRA Adapter |
|---|---:|---:|
| 验证集 loss | 0.5198 | 0.2830 |
| 无约束动作有效数 | 7/18 | 13/18 |
| 无约束检测调用 | 0/6 | 6/6 |
| 无约束检索调用 | 5/6 | 6/6 |
| 无约束证据选择 | 2/6 | 1/6 |
| 受约束完整流程 | 6/6 | 6/6 |
| 完整流程测量值保留 | 6/6 | 6/6 |

训练耗时约 166 秒；训练及评测共约 764 秒；峰值 allocated 显存 6.45 GiB。

**结论：调用流程有所改善，但证据选择仍然薄弱，且该项发生退步。Adapter 保留为实验选项，不替换默认模型。**

这是小规模合成数据实验，不能证明真实芯片缺陷检测或制造根因识别能力。ZIP 保存 Adapter、数据集、原始输出及校验清单。

In [4]:
# Persist a small audit output separately from the large ZIP download.
import json, subprocess, sys
from xsim_chip_analysis.assistant.reports import normalize_report
source_report = 'docs/results/volume_bridge_2026-10-02/volume_inspection_report.json'
command = [sys.executable, '-u', '-m', 'xsim_chip_analysis.assistant.cli', '--report', source_report, '--model', 'Qwen/Qwen2.5-1.5B-Instruct', '--revision', '989aa7980e4cf806f80c7fef2b1adb7bc71aa306', '--adapter', 'outputs/lora_pilot/adapter', '--strict', '--output-dir', 'outputs/lora_integration']
with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
    for line in process.stdout:
        print(line, end='', flush=True)
    code = process.wait()
if code:
    raise subprocess.CalledProcessError(code, command)
integration = json.loads((repo / 'outputs/lora_integration/assistant_result.json').read_text())
assert integration['mode'] == 'llm_agent'
assert integration['evidence'] == normalize_report(json.loads((repo / source_report).read_text()))
audit = {'training': training, 'evaluation': evaluation, 'original_pipeline_integration': integration}
print('AUDIT_JSON_BEGIN\n' + json.dumps(audit, ensure_ascii=False, indent=2) + '\nAUDIT_JSON_END', flush=True)


2026-10-04 15:58:59.714584: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
{"mode": "llm_agent", "components": 82, "accepted_interpretations": 6, "output_dir": "outputs/lora_integration"}
AUDIT_JSON_BEGIN
{
  "training": {
    "base_model": "Qwen/Qwen2.5-1.5B-Instruct",
    "base_revision": "989aa7980e4cf806f80c7fef2b1adb7bc71aa306",
    "gpu": "Tesla T4",
    "python": "3.13.15",
    "packages": {
      "torch": "2.11.0+cu130",
      "transformers": "4.57.6",
      "peft": "0.17.1",
      "accelerate": "1.15.0",
      "lm-format-enforcer": "0.11.3"
    },
    "git_commit": "11f5f816c37aff0217bc5a9b147597143fb6f1cb",
    "settings": {
      "output_dir": "outputs/lora_pilot/trainer",
      "overwrite_output_dir": false,
      "do_train": false,
